# K-Fold v6 — v30_optimize (cross-validate model deploy) — cross-population (Elderly / Young / Both)

K-fold subject-independent cho **kiến trúc v30_optimize** (Separable-CNN + head GAP+GMP+Flatten giữ cấu trúc thời gian, train thuần KHÔNG KD). Khung kế thừa K-Fold v4 (cùng pipeline `decimate(q=2)`, windowing `v3kf`, cache `cache_kfold_v3`) — chỉ thay `build_model` + monitor `val_accuracy`. 5 kịch bản: S1 Elderly / S2 Young / S3 train SE→test SA / S4 train SA→test SE / S5 Both. Kết quả lưu `KFold_Results_v6/`.

> Lưu ý: pipeline k-fold dùng `decimate(q=2)` (khác `iloc[::2]` của bản deploy) — số dùng để đánh giá **robustness** của kiến trúc, nhất quán nội bộ. Báo cáo Fall recall chủ yếu ở S5_Both + S3/S4 (S1 Elderly thiếu nhãn Fall).

In [ ]:
# === Server (env conda) — Keras 2 legacy cho khop train_v30 ===
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
!pip install -q tf-keras

import shutil

ZIP_PATH  = '/media/data3/users/hungvm/dataset/sisfall/archive.zip'
WORKSPACE = '/media/data3/users/hungvm/dataset/sisfall/workspace'

need_extract = (not os.path.exists(WORKSPACE)) or (len(os.listdir(WORKSPACE)) == 0)
if need_extract:
    os.makedirs(WORKSPACE, exist_ok=True)
    shutil.unpack_archive(ZIP_PATH, WORKSPACE)
    print("Giai nen hoan tat!")
else:
    print("Workspace da ton tai!")
print("Noi dung WORKSPACE:", os.listdir(WORKSPACE))

# 2. Tiền xử lý (Step 1)
Accel→g (`raw*32/8192`); gyro→dps (`raw*4000/65536`, KHÔNG ×π/180); downsample 200→100Hz bằng **`decimate`** (lọc chống aliasing zero-phase — "downsample xịn").

In [ ]:
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from scipy.signal import decimate

INPUT_DIR  = Path(WORKSPACE) / 'SisFall_dataset'
OUTPUT_DIR = Path(WORKSPACE) / 'SisFall_dataset_Processed_v3kf'

target_har = {'D01', 'D02', 'D03', 'D04', 'D05', 'D07', 'D08', 'D11', 'D12', 'D13', 'D15', 'D17', 'D18', 'D19'}
ACCEL_SCALE = 32.0 / 8192.0
GYRO_SCALE = 4000.0 / 65536.0   # raw -> dps (KHONG nhan pi/180), khop firmware

all_files = list(INPUT_DIR.rglob('*.txt'))
files_to_process = [f for f in all_files if f.name.startswith('F') or f.name[:3] in target_har]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for file_path in tqdm(files_to_process, desc="Preprocessing v3kf"):
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()

        data = []
        for line in lines:
            line = line.strip().rstrip(';')
            if not line: continue
            parts = line.split(',')
            if len(parts) >= 6:
                data.append([float(x) for x in parts[:6]])

        if not data: continue
        df = pd.DataFrame(data, columns=['ax', 'ay', 'az', 'gx', 'gy', 'gz'])

        df[['ax', 'ay', 'az']] *= ACCEL_SCALE
        df[['gx', 'gy', 'gz']] *= GYRO_SCALE

        # Downsample 200Hz -> 100Hz CO loc thong thap chong aliasing (zero-phase, khong lech peak).
        # decimate(q=2) tu ap bo loc Chebyshev bac 8 truoc khi bo mau -> giu dung dang dinh impact.
        if len(df) > 27:  # decimate can du mau cho bo loc IIR (padlen)
            arr = decimate(df.to_numpy(), q=2, axis=0, ftype='iir', zero_phase=True)
            df = pd.DataFrame(arr.astype(np.float32), columns=df.columns)
        else:
            df = df.iloc[::2, :].reset_index(drop=True)

        rel_path = file_path.relative_to(INPUT_DIR)
        out_file = OUTPUT_DIR / rel_path.parent / (file_path.stem + '.csv')
        out_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_file, index=False)
    except Exception as e:
        pass
print("Hoan tat Step 1!")

# 3. Cắt Window (Step 2) — Trans theo SỰ KIỆN gyro (KHỚP train_v30)
Mỗi vùng gyro rolling-RMS > 20 dps liên tục = 1 sự kiện transition → cắt 1 cửa sổ 2s căn vào đỉnh. Idle = xa mọi sự kiện gyro ≥ 80 mẫu. D18/D19 = 1 Trans tại đỉnh gyro.

In [ ]:
import numpy as np
from pathlib import Path

WINDOW_OUTPUT_DIR = Path(WORKSPACE) / 'SisFall_dataset_Windowed_v3kf'
GYRO_RMS_TH = 20.0     # 20 dps (gyro trong CSV da o dps)

def rolling_rms(x, w=21):
    return np.sqrt(np.convolve(x**2, np.ones(w)/w, mode='same'))

def extract_window(df, center, window_size=200):
    start, end = center - window_size // 2, center + window_size // 2
    if start < 0: start, end = 0, window_size
    if end > len(df): end, start = len(df), max(0, len(df) - window_size)
    return df.iloc[start:end]

def classify_idle(win_df):
    ax_m, ay_m, az_m = win_df['ax'].abs().mean(), win_df['ay'].abs().mean(), win_df['az'].abs().mean()
    return 'Idle_StandSit' if (ay_m > ax_m and ay_m > az_m) else 'Idle_Lie'

def gyro_event_peaks(df, min_len=10, gap=10):
    """Moi vung gyro-RMS > nguong lien tuc = 1 su kien transition; tra ve dinh cua moi su kien."""
    gmag = np.sqrt(df['gx'].values**2 + df['gy'].values**2 + df['gz'].values**2)
    grms = rolling_rms(gmag, 21)
    idx = np.where(grms > GYRO_RMS_TH)[0]
    peaks = []
    if len(idx):
        for grp in np.split(idx, np.where(np.diff(idx) > gap)[0] + 1):
            if len(grp) >= min_len:
                peaks.append(int(grp[0] + np.argmax(grms[grp[0]:grp[-1]+1])))
    return peaks

all_processed = list(OUTPUT_DIR.rglob('*.csv'))
trans_har    = {'D07','D08','D11','D12','D13','D15','D17'}   # ngoi/dung/nam -> Trans + Idle
stumble_jump = {'D18','D19'}                                  # vap/nhay -> 1 Trans tai dinh gyro
cont_har     = {'D01','D02','D03','D04','D05'}                # di/chay lien tuc

windows_generated = 0
for f in tqdm(all_processed, desc="Windowing v3kf (gyro-Trans)"):
    df = pd.read_csv(f)
    if len(df) < 200: continue
    filename, prefix = f.stem, f.stem[:3]
    out_dir = WINDOW_OUTPUT_DIR / f.parent.name
    out_dir.mkdir(parents=True, exist_ok=True)

    if filename.startswith('F'):
        svm = np.sqrt(df['ax']**2 + df['ay']**2 + df['az']**2)
        peak_idx = int(svm.values.argmax())
        for w_idx, shift in enumerate([-60,-30,0,30,60]):
            win = extract_window(df, peak_idx+shift)
            if len(win)==200:
                win.to_csv(out_dir/f"{filename}_Fall_W{w_idx:03d}.csv", index=False); windows_generated+=1

    elif prefix in stumble_jump:
        gmag = np.sqrt(df['gx'].values**2 + df['gy'].values**2 + df['gz'].values**2)
        peak = int(rolling_rms(gmag,21).argmax())
        win = extract_window(df, peak)
        if len(win)==200:
            win.to_csv(out_dir/f"{filename}_Trans_P000.csv", index=False); windows_generated+=1

    elif prefix in trans_har:
        peaks = gyro_event_peaks(df)
        for i, p in enumerate(peaks):
            win = extract_window(df, p)
            if len(win)==200:
                win.to_csv(out_dir/f"{filename}_Trans_E{i:03d}.csv", index=False); windows_generated+=1
        for start in range(0, len(df)-200+1, 100):
            center = start + 100
            if all(abs(center - p) >= 80 for p in peaks):
                win = df.iloc[start:start+200]
                win.to_csv(out_dir/f"{filename}_{classify_idle(win)}_W{start//100:03d}.csv", index=False); windows_generated+=1

    elif prefix in cont_har:
        for start in range(0, len(df)-200+1, 100):
            win = df.iloc[start:start+200]
            label = "Walk" if prefix in {'D01','D02','D05'} else "Run"
            win.to_csv(out_dir/f"{filename}_{label}_W{start//100:03d}.csv", index=False); windows_generated+=1

print(f"Hoan tat Windowing v3kf! Tong cua so: {windows_generated}")

# 4. Import + cấu hình K-Fold

In [ ]:
import pickle, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import KFold
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from scipy import stats
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv1D, SeparableConv1D, BatchNormalization, Activation,
                                     MaxPooling1D, GlobalAveragePooling1D, Flatten, Concatenate, Dropout, Dense)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

DRIVE_OUT = Path('/media/data3/users/hungvm/dataset/sisfall/KFold_Results_v6')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print(f"[*] Ket qua se duoc luu tai: {DRIVE_OUT}")

# 5. K-Fold Data Manager (gom theo Subject, Subject-Independent)
`get_data_for_subjects` trả về **dữ liệu RAW** (chưa normalize). Preprocessing được áp **sau khi augment Trans** bên trong `train_and_evaluate` — đúng thứ tự của train_v30 (augment biên độ trên raw → mới clip ±500/500).

In [ ]:
class KFoldDataManager:
    def __init__(self, windowed_dir, cache_dir, class_names):
        self.windowed_dir = Path(windowed_dir)
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.class_names = class_names

    def parse_filename_info(self, filename):
        if '_Trans_' in filename: return 'Trans'
        if '_StandSit_' in filename or '_Lie_' in filename: return 'Idle'
        prefix = filename[:3]
        if prefix in ['D01', 'D02', 'D05', 'D06']: return 'Walk'
        if prefix in ['D03', 'D04']: return 'Run'
        if prefix.startswith('F'): return 'Fall'
        return None

    def load_single_csv(self, file_path):
        try:
            filename = file_path.stem
            subject_id = filename.split('_')[1]
            label_str = self.parse_filename_info(filename)
            if label_str is None: return None
            label = self.class_names.index(label_str)
            df = pd.read_csv(file_path)
            if len(df) != 200: return None
            return df.to_numpy(), label, subject_id
        except: return None

    def load_and_group_by_subject(self):
        cache_file = self.cache_dir / 'subject_data.pkl'
        if cache_file.exists():
            print("[*] Loading subject data from cache...")
            with open(cache_file, 'rb') as f:
                return pickle.load(f)

        all_files = list(self.windowed_dir.rglob('*.csv'))
        if len(all_files) == 0:
            raise FileNotFoundError(f"LOI: Khong tim thay file CSV nao trong {self.windowed_dir}")

        subject_data = {}
        with ThreadPoolExecutor(max_workers=8) as executor:
            for res in tqdm(executor.map(self.load_single_csv, all_files), total=len(all_files), desc="Loading CSV"):
                if res is None: continue
                data, label, sub = res
                if sub not in subject_data:
                    subject_data[sub] = {'X': [], 'y': []}
                subject_data[sub]['X'].append(data)
                subject_data[sub]['y'].append(label)

        for sub in subject_data:
            subject_data[sub]['X'] = np.array(subject_data[sub]['X'], dtype=np.float32)
            subject_data[sub]['y'] = np.array(subject_data[sub]['y'], dtype=np.int32)

        with open(cache_file, 'wb') as f:
            pickle.dump(subject_data, f)
        print(f"[*] Da tai du lieu cho {len(subject_data)} subjects.")
        return subject_data

    def apply_preprocessing(self, X):
        # v30: accel clip +-8g /8 ; gyro (dps) clip +-500 /500 (KHOP firmware) -- 6 kenh
        if X.ndim == 3 and X.shape[0] > 0:
            X[:, :, 0:3] = np.clip(X[:, :, 0:3], -8.0, 8.0) / 8.0
            X[:, :, 3:6] = np.clip(X[:, :, 3:6], -500.0, 500.0) / 500.0
        return X

    def get_data_for_subjects(self, subject_data, subjects):
        # Tra ve RAW (chua preprocessing) -> augment + normalize lam ben train_and_evaluate
        X_list, y_list = [], []
        for sub in subjects:
            if sub in subject_data:
                X_list.append(subject_data[sub]['X'])
                y_list.append(subject_data[sub]['y'])
        if len(X_list) == 0:
            return np.array([]), np.array([])
        X = np.concatenate(X_list, axis=0).astype(np.float32)
        y = np.concatenate(y_list, axis=0)
        return X, y

# 6. Kiến trúc Model — **TCN v22** (dilated + SE) — cross-val v30_tcn

In [ ]:
def build_model(input_shape=(200, 6), n_classes=5):
    # v30_optimize: Separable-CNN toi uu ESP-NN, head GAP+GMP+Flatten (giu cau truc thoi gian) ~26.6k params.
    #   = kien truc kd2 train THUAN (khong KD/teacher). 2 SepConv k3/tang dau + kenh 32/48/64/96.
    inp = Input(shape=input_shape)
    x = Conv1D(32, 3, strides=2, padding='same', use_bias=False)(inp)        # 200->100
    x = BatchNormalization()(x); x = Activation('relu6')(x)
    x = SeparableConv1D(48, 3, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x); x = Activation('relu6')(x)
    x = SeparableConv1D(48, 3, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x); x = Activation('relu6')(x)
    x = MaxPooling1D(2)(x)                                                    # 100->50
    x = SeparableConv1D(64, 3, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x); x = Activation('relu6')(x)
    x = SeparableConv1D(64, 3, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x); x = Activation('relu6')(x)
    x = MaxPooling1D(2)(x)                                                    # 50->25
    x = SeparableConv1D(96, 3, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x); x = Activation('relu6')(x)
    x = MaxPooling1D(2)(x)                                                    # 25->12
    a = GlobalAveragePooling1D()(x)
    m = MaxPooling1D(pool_size=int(x.shape[1]))(x); m = Flatten()(m)
    f = Flatten()(x)
    x = Concatenate()([a, m, f])
    x = Dropout(0.3)(x)
    outputs = Dense(n_classes, activation='softmax')(x)
    model = Model(inp, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                  metrics=['accuracy'])
    return model

# 7. Hàm Train & Evaluate — augment Trans ×0.9/×1.1, class-weight Fall×3 Trans×0.55 (KHỚP train_v30)

In [ ]:
def augment_trans(X, y, class_names):
    # v30: chi nhan bien do x0.9 va x1.1 cho Trans, TREN RAW (truoc preprocessing)
    if len(X) == 0:
        return X, y
    ti = class_names.index('Trans')
    tmask = np.where(y == ti)[0]
    if len(tmask) > 0:
        Xt = X[tmask]
        X = np.concatenate([X, (Xt*0.9).astype(np.float32), (Xt*1.1).astype(np.float32)], axis=0)
        y = np.concatenate([y, np.full(len(Xt), ti, y.dtype), np.full(len(Xt), ti, y.dtype)])
        p = np.random.RandomState(42).permutation(len(X))
        X, y = X[p], y[p]
    return X, y

def train_and_evaluate(X_train, y_train, X_test, y_test, run_name, class_names, epochs=100):
    print(f"\n{'='*50}\nRunning: {run_name}\n{'='*50}")
    print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

    # 1. Augment Trans (raw) -> 2. Preprocessing (clip+normalize) -- dung thu tu v30
    X_train, y_train = augment_trans(X_train, y_train, class_names)
    X_train = data_manager.apply_preprocessing(X_train)
    X_test  = data_manager.apply_preprocessing(X_test)

    # 3. Class weights: balanced * {Fall:3, Trans:0.55, Idle:1}
    if len(y_train) > 0:
        vals = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        c_weights = dict(zip(np.unique(y_train), vals))
        for nm, mod in {'Fall': 3.0, 'Trans': 0.55, 'Idle': 1.0}.items():
            idx = class_names.index(nm)
            if idx in c_weights: c_weights[idx] *= mod
    else:
        c_weights = None

    y_train_oh = to_categorical(y_train, num_classes=5)
    model = build_model()
    model_path = DRIVE_OUT / f"model_{run_name}.keras"
    callbacks = [
        EarlyStopping(monitor='val_accuracy', mode='max', patience=15, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_accuracy', mode='max', factor=0.5, patience=5, min_lr=1e-6, verbose=0),
        ModelCheckpoint(filepath=str(model_path), monitor='val_accuracy', mode='max', save_best_only=True, verbose=0),
    ]
    history = model.fit(
        X_train, y_train_oh,
        validation_split=0.1,
        epochs=epochs, batch_size=64,
        class_weight=c_weights,
        callbacks=callbacks, verbose=0
    )

    # Eval (threshold Fall 0.25)
    y_pred_probs = model.predict(X_test, batch_size=256, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_pred[y_pred_probs[:, 4] >= 0.25] = 4

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    all_labels = [0, 1, 2, 3, 4]
    report_dict = classification_report(y_test, y_pred, labels=all_labels,
                                        target_names=class_names, output_dict=True, zero_division=0)
    print(f"Accuracy: {acc:.4f} | F1-Score (Macro): {f1:.4f}")

    cm = confusion_matrix(y_test, y_pred, labels=all_labels)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix: {run_name}')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.tight_layout(); plt.savefig(DRIVE_OUT / f'CM_{run_name}.png'); plt.close()

    tf.keras.backend.clear_session()
    return {'acc': float(acc), 'f1': float(f1), 'report': report_dict}

# 8. Cấu hình dữ liệu & các kịch bản K-Fold

In [ ]:
class_names = ['Walk', 'Run', 'Idle', 'Trans', 'Fall']
data_manager = KFoldDataManager(str(Path(WORKSPACE) / 'SisFall_dataset_Windowed_v3kf'),
                                str(Path(WORKSPACE) / 'cache_kfold_v3'), class_names)

subject_data = data_manager.load_and_group_by_subject()

sa_subjects = [f"SA{i:02d}" for i in range(1, 24)]
se_subjects = [f"SE{i:02d}" for i in range(1, 16)]
sa_subjects = [s for s in sa_subjects if s in subject_data]
se_subjects = [s for s in se_subjects if s in subject_data]

results = {}
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

### Kịch bản 1: Elderly Only (5-Fold) — *lưu ý: SE thiếu nhãn Fall*

In [ ]:
res_s1 = []
print("\n" + "="*50 + "\n=== Scenario 1: Elderly Only (K-Fold) ===\n" + "="*50)
for i, (train_idx, test_idx) in enumerate(kf.split(se_subjects)):
    train_subs = [se_subjects[idx] for idx in train_idx]
    test_subs  = [se_subjects[idx] for idx in test_idx]
    X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, train_subs)
    X_te, y_te = data_manager.get_data_for_subjects(subject_data, test_subs)
    res_s1.append(train_and_evaluate(X_tr, y_tr, X_te, y_te, f"S1_Elderly_Fold{i+1}", class_names))
results['S1_Elderly'] = res_s1

### Kịch bản 2: Young Only (5-Fold)

In [ ]:
res_s2 = []
print("\n" + "="*50 + "\n=== Scenario 2: Young Only (K-Fold) ===\n" + "="*50)
for i, (train_idx, test_idx) in enumerate(kf.split(sa_subjects)):
    train_subs = [sa_subjects[idx] for idx in train_idx]
    test_subs  = [sa_subjects[idx] for idx in test_idx]
    X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, train_subs)
    X_te, y_te = data_manager.get_data_for_subjects(subject_data, test_subs)
    res_s2.append(train_and_evaluate(X_tr, y_tr, X_te, y_te, f"S2_Young_Fold{i+1}", class_names))
results['S2_Young'] = res_s2

### Kịch bản 3: Train Elderly → Test Young

In [ ]:
print("\n" + "="*50 + "\n=== Scenario 3: Train Elderly -> Test Young ===\n" + "="*50)
X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, se_subjects)
X_te, y_te = data_manager.get_data_for_subjects(subject_data, sa_subjects)
results['S3_TrainSE_TestSA'] = train_and_evaluate(X_tr, y_tr, X_te, y_te, "S3_TrainSE_TestSA", class_names)

### Kịch bản 4: Train Young → Test Elderly  *(kịch bản chủ lực)*

In [ ]:
print("\n" + "="*50 + "\n=== Scenario 4: Train Young -> Test Elderly ===\n" + "="*50)
X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, sa_subjects)
X_te, y_te = data_manager.get_data_for_subjects(subject_data, se_subjects)
results['S4_TrainSA_TestSE'] = train_and_evaluate(X_tr, y_tr, X_te, y_te, "S4_TrainSA_TestSE", class_names)

### Kịch bản 5: Train Both → Test Both (5-Fold)

In [ ]:
res_s5 = []
print("\n" + "="*50 + "\n=== Scenario 5: Train Both -> Test Both (K-Fold) ===\n" + "="*50)
kf_sa = list(kf.split(sa_subjects))
kf_se = list(kf.split(se_subjects))
for i in range(n_splits):
    train_sa_idx, test_sa_idx = kf_sa[i]
    train_se_idx, test_se_idx = kf_se[i]
    train_subs = [sa_subjects[idx] for idx in train_sa_idx] + [se_subjects[idx] for idx in train_se_idx]
    test_subs  = [sa_subjects[idx] for idx in test_sa_idx] + [se_subjects[idx] for idx in test_se_idx]
    X_tr, y_tr = data_manager.get_data_for_subjects(subject_data, train_subs)
    X_te, y_te = data_manager.get_data_for_subjects(subject_data, test_subs)
    res_s5.append(train_and_evaluate(X_tr, y_tr, X_te, y_te, f"S5_Both_Fold{i+1}", class_names))
results['S5_Both'] = res_s5

# 9. Kiểm định thống kê + lưu kết quả + tóm tắt

In [ ]:
print("\n" + "="*50 + "\n=== Statistical Significance Test ===\n" + "="*50)

f1_s1 = [r['f1'] for r in results['S1_Elderly']]
f1_s2 = [r['f1'] for r in results['S2_Young']]
f1_s5 = [r['f1'] for r in results['S5_Both']]

t1, p1 = stats.ttest_ind(f1_s1, f1_s2)
print(f"H1 (Elderly vs Young F1): T-stat={t1:.4f}, p-value={p1:.4f}")
print("-> Co y nghia thong ke." if p1 < 0.05 else "-> KHONG co y nghia thong ke.")

t2, p2 = stats.ttest_ind(f1_s5, f1_s1)
print(f"\nH2 (Both vs Elderly F1): T-stat={t2:.4f}, p-value={p2:.4f}")
print("-> Cai thien co y nghia." if p2 < 0.05 else "-> Cai thien KHONG co y nghia.")

# Tom tat: acc/f1 trung binh + Fall recall moi kich ban
def fall_recall(r):
    fr = r['report'].get('Fall', {})
    return fr.get('recall', 0.0), fr.get('support', 0)

print("\n" + "="*50 + "\n=== TOM TAT (mean acc / mean f1 / Fall recall) ===\n" + "="*50)
for key in ['S1_Elderly', 'S2_Young', 'S3_TrainSE_TestSA', 'S4_TrainSA_TestSE', 'S5_Both']:
    val = results[key]
    folds = val if isinstance(val, list) else [val]
    acc = np.mean([f['acc'] for f in folds])
    f1m = np.mean([f['f1'] for f in folds])
    frs = [fall_recall(f) for f in folds]
    fr_mean = np.mean([r for r, s in frs])
    sup_tot = int(np.sum([s for r, s in frs]))
    print(f"{key:20s} acc={acc:.4f}  f1={f1m:.4f}  Fall_recall={fr_mean:.4f} (support={sup_tot})")

final_output = DRIVE_OUT / 'final_metrics.json'
with open(final_output, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)
print(f"\n[*] Da luu ket qua vao: {final_output}")